### LAVA2D

Lava2d is a depth-average flow model, unique in the fact that it incorporates variable temperatures efficently for quick and realistic simulations. 

### **This model is not sanctioned for forecasting work by USGS or the developers**: Do not use in those contexts!

In [ ]:
import xarray as xr
import netCDF4 as nc
import matplotlib.pyplot as plt
import rasterio as rio
import cartopy.crs as ccrs
from pyproj import Transformer
import subprocess
import numpy as np
import sys
sys.path.insert(0,"/home/jovyan/shared/Libraries")
import victor
import utm

In [ ]:
#dem = "./MaunaLoa.tif"
dem = "/home/jovyan/shared/DEMs/Iceland_20mx20m_cropped.tif"
x = rio.open(dem)

### After importing the necessary Python libraries, we must specify a DEM to use. 

Only TIFF/geotiff files are valid, and must have latitude/longitudinal georeferencing.

In [ ]:
bounds = x.bounds
crs = x.crs
transformer = Transformer.from_crs(crs, "EPSG:4326")
transformer2 = Transformer.from_crs("EPSG:4326", crs)
lower_left = transformer.transform(bounds.left,bounds.bottom)
upper_right = transformer.transform(bounds.right,bounds.top)
print("Your boundary coordinates, for reference, is", lower_left, ", ", upper_right, ".")

### Please enter the resolution you would like to model at.

As a depth-averaged finite volume model, calculations can often take a long time, especially if the provided topography is very high detail/erratic. Below, you can specify the scaling and smoothing parameters. 

Note: Lava2d adjusts the resolution in powers of 2.

In [ ]:
# initially set to default resolution
dx_desired = x.res[0]*4          # meters
    
smooth_n_times = 1

### Please specify the source coodinates of the vent

In [ ]:
#Mauna Loa: 
# lon_src = -155.592
# lat_src = 19.472

#Iceland
lon_src = -22.4
lat_src = 63.9
lonlat = transformer2.transform(lat_src,lon_src)

### Now, lets confirm our use of the DEM and the vent locations

In [ ]:
victor.plot_dem(dem, markercoords=np.array([lonlat[0], lonlat[1]]))

### Now, please set the properties of the erupting lava (at vent)

In [ ]:
vent_temperature_c = 1200

viscosity_melt_vent = 1000

crystal_percent = .02

volume = 1e8

### Now, we can set the constants for the lava

In [ ]:
liquid_density = 2700

porosity = .2

lava_specific_heat = 1500

lava_diffusivity = 5e-7

lava_conductivity = None

lava_emissivity = .95

### Below, enter rheological properties

In [ ]:
phi_max_crystal_packing = .6

max_cryst_rate = 1e-4

yield_strength_crust = 1e4

T_core_T_vent_equal = True

### Select ambient temperature values below

In [ ]:
ground_temperature = 300

atm_temperature = 300

h_conv = 50

### Please enter numerical parameters

In [ ]:
# Set minimum/maximum model-clock ratio =

efficiency_min = 0

efficiency_max = 10000

cfl_max = 0.5

dt_max = 5

freezable_fraction = 1.0

min_thickness = 0.1

### Set the max simulation time, and intervals to output data

In [ ]:
max_time_hrs = 3

out_times = [0.5, 1, 1.5, 2]

In [ ]:
seconds = max_time_hrs*3600

dis = volume/seconds

discharge = [dis, dis]

### Specify vent eruption timing and relative location

Please include at least two events in the array.

For reference, the discharge value in the vents files are the DRE instantaneous eruption rate (units: $m^3/s$).  The source term is generated at some time point by linearly interpolating these values.  

In [ ]:
times = [0, 10000]

xy0 = [[0, 0],[0, 0]]

xy1 = [[100, -75],[100, -75]]

width = [20, 20]

times = np.array(times)
xy0 = np.array(xy0)
xy1 = np.array(xy1)
width = np.array(width)
discharge = np.array(discharge)

In [ ]:
inp = f"""import sim
#
#-------------------------------------------------------------------------------
sim.set_topo( # set DEM info
    path_to_dem_file    = ("{dem}"),
    Lon_SRC             = {lon_src}, # source longitude
    Lat_SRC             = {lat_src},    # source latitude
    Lon_LowerLeft       = {lower_left[1]}, # bounding box: lower-left longitude
    Lat_LowerLeft       = {lower_left[0]}, # bounding box: lower-left latitude
    Lon_UpperRight      = {upper_right[1]}, # bounding box: upper-right longitude
    Lat_UpperRight      = {upper_right[0]},   # bounding box: upper-right latitude
    fill_level          = 0.0,     # fill topography up to fill_level (m.a.s.l.)
    dx_desired          = {dx_desired},          # meters
    smooth_n_times      = {smooth_n_times}
    )

sim.set_init( # set initialization type and file
    init_type = None,  # None or 'prior_model' currently supported
    init_file = None  # only specify if init_type = 'prior_model'
    )
#
#-------------------------------------------------------------------------------
sim.set_vent_props( # set lava properties @ vents
    temperature_vent_C  = {vent_temperature_c}, # deg C
    viscosity_melt_vent = {viscosity_melt_vent}, # Pa s
    cryst_vent          = {crystal_percent}  # erupted crystal fraction
    )
#
#-------------------------------------------------------------------------------
sim.set_lava_props( # set lava properties throughout
    liquid_density      = {liquid_density},   # kg m-3
    porosity            = {porosity},
    lava_specific_heat  = {lava_specific_heat},   # J kg-1 K-1
    lava_diffusivity    = {lava_diffusivity}, # m2 s-1
    lava_conductivity   = {lava_conductivity},    # W m-1 K-1
    lava_emissivity     = {lava_emissivity}
    )
#
#-------------------------------------------------------------------------------
# using Avrami n = 4
sim.set_rheo( # set rheological properties
    phi_max                 = {phi_max_crystal_packing}, # max crystal packing fraction (could be 0.6 e.g., Marsh 1981; Pinkerton and Stevenson 1992), # could be higher (Cashman et al., 1999)
    max_cryst_rate          = {max_cryst_rate}, # s-1 # max crystalization rate: max d(phi)/dt
    yield_strength_crust    = {yield_strength_crust}, # Pa
    T_core_T_vent_equal     = {T_core_T_vent_equal}  # core temperature equals vent temperature
    )
#
#-------------------------------------------------------------------------------
sim.set_ambient( # set ambient properties
    ground_temperature  = {ground_temperature}, # K
    atm_temperature     = {atm_temperature}, # K
    h_conv              = {h_conv},   # W m-2 K-1
    )
#
#-------------------------------------------------------------------------------
sim.set_numerics( # set numerical method details
    efficiency_min      = {efficiency_min}, # minmum allowable model-clock ratio
    efficiency_max      = {efficiency_max},  # maximum allowable model-clock ratio
    cfl_max             = {cfl_max},
    dt_max              = {dt_max},
    fraction_to_freeze  = {freezable_fraction},  # fraction of freezable lava per time-step
    tiny_flow           = {min_thickness},  # min thickness of lava (m)
    )
#
#-------------------------------------------------------------------------------
sim.set_runtime(
    max_iter = None, #one of them can be none, so the default value for both is none, if both none run till killed
    max_time_hr = {max_time_hrs},
    out_times = {out_times}, # hr, list of intermediate output times
    run_to_ocean = True
    )
#
#-------------------------------------------------------------------------------
sim.set_output( # where to store out.nc?
    path_out =  ('./outputs')
    )
#
#-------------------------------------------------------------------------------
sim.set_source( # set vent/fissure info: where is vent_nn.txt located?
    path_to_vent_files      = ('./example_vents')
    ) # all vent files must be named vent_01.txt, vent_02.txt, etc
#
#-------------------------------------------------------------------------------
#start simulation
sim.run()
#
#-------------------------------------------------------------------------------
#"""
f = open("input.py","w")
f.write(inp)
f.close()

In [ ]:
vents = f"""time	x0	y0	x1	y1	width	discharge"""
for time in range(len(times)):
    vents += f"""\n{times[time]}	{xy0[time,0]}	{xy0[time,1]}	{xy1[time,0]}	{xy1[time,1]}	{width[time]}	{discharge[time]}"""
# {time}	{x0}	{y0}	{x1}	{y1}	{width}	{discharge}"""
g = open("./example_vents/vent_01.txt","w")
g.write(vents)
g.close()

### Executing the cell below will run the model.

**NB**: Based on the discharge rate, granularity/smoothing, and other parameters, this may take a while.

In [ ]:
subprocess.run("python input.py",shell=True)

### Below, select a timestep based on the output times entered above

In [ ]:
print(f"For reference, the output times times are {out_times}. If you want to select the final output, enter -1.")

In [ ]:
chosen_time = .5

In [ ]:
import rioxarray as rxr
if chosen_time == -1:
    output_file = "out.nc"
else:
    chosen_time = float(chosen_time)
    output_file = f"out.T+{str(chosen_time).zfill(5)}hr.nc"
data = xr.open_dataset(f"outputs/{output_file}",group="DATA/PHYSICS",decode_coords="all")
data = data.where(data!=0, np.nan)

coord = xr.open_dataset(f"outputs/{output_file}",group="DATA", decode_coords="all")
lat = coord.lat[:,0]
lon = coord.lon[0,:]
data['rows'] = lat.values
data['cols'] = lon.values

In [ ]:
data = data.rename({"rows": "y", "cols": "x"})
repeated_arr_x = np.repeat(data.y.values[0], len(data.x))
repeated_arr_y = np.repeat(data.x.values[0], len(data.y))

utm1 = transformer2.transform(repeated_arr_x,data.x.values)
utm2 = transformer2.transform(data.y.values,repeated_arr_y)
data['x'] = utm1[0]
data['y'] = utm2[1]
data.lava_thickness_total.rio.to_raster("lava2d.tif")

In [ ]:
fig, ax = plt.subplots()
victor.plot_flow(dem, "lava2d.tif", axes=ax)
ax.ticklabel_format(useOffset=False)